# Config
Imports and table names used by the other notebooks, plus one-time setup of the control and audit tables.

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
CATALOG = "moh_raz" 
SCHEMA  = "raz_casemix_stg"
CONTROL_TABLE = f"{CATALOG}.{SCHEMA}.CSS_DELETION_AUDIT_CONTROL_TEST"
AUDIT_TABLE = f"{CATALOG}.{SCHEMA}.CSS_DELETION_AUDIT_TEST"

## Control table
Tracks which tables to monitor and the last Delta version already processed for each.

### Temp - remove in prod

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {CONTROL_TABLE}")
spark.sql(f"DROP TABLE IF EXISTS {AUDIT_TABLE}")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CONTROL_TABLE} (
    catalog_name           STRING NOT NULL,
    schema_name            STRING NOT NULL,
    table_name             STRING NOT NULL,
    primary_key_columns    STRING NOT NULL,
    is_active              INT,
    last_processed_version BIGINT,
    created_at              TIMESTAMP,
    updated_at              TIMESTAMP
)
USING DELTA; """)


%md
### Temp - remove in prod

In [0]:
%sql
INSERT INTO moh_raz.raz_casemix_stg.CSS_DELETION_AUDIT_CONTROL_TEST
(
    catalog_name,
    schema_name,
    table_name,
    primary_key_columns,
    is_active,
    last_processed_version,
    created_at,
    updated_at
)
VALUES
(
    'moh_raz',
    'raz_casemix_stg',
    'patient_details_test',
    'customer_id',
    1,
    1,
    current_timestamp(),
    current_timestamp()
);

## Audit table
Stores one row per deleted record. `table_name + primary_id + commit_version` is the natural key used for de-duplication

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {AUDIT_TABLE} (
    audit_id          STRING,
    catalog_name      STRING,
    schema_name       STRING,
    table_name        STRING,
    primary_id        STRING,
    deleted_by        STRING,
    deleted_at        TIMESTAMP,
    commit_version    BIGINT,
    commit_timestamp  TIMESTAMP,
    operation         STRING,
    loaded_at         TIMESTAMP
)
USING DELTA;
          """)